# Problem 3: Extended Kalman Filter for Robot Localization
All of your code goes in `P3_ekf.py`. The landmark positions are known; the filter estimates the robot pose from the controls and noisy measurements of the landmarks in the robot frame.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from P3_ekf import dynamics, measurement, jacobian_G, jacobian_H, ekf_predict, ekf_update, run_ekf
from utils import plot_error_ellipse

os.makedirs("plots", exist_ok=True)
data = np.load("data/filter_data.npz")
time, landmarks, us, dt = data["time"], data["landmarks"], data["controls"], float(data["dt"])
poses, zs = data["ekf_poses"], data["ekf_z"]
mu0, Sigma0, R, Q = data["ekf_mu0"], data["ekf_Sigma0"], data["ekf_R"], data["ekf_Q"]
print("time steps:", len(us), " R = %.4f I, Q = %.2f I" % (R[0, 0], Q[0, 0]))

## Checking the Jacobians
Both Jacobians are compared with finite differences at a random pose. The errors should be tiny (about 1e-8 or smaller).

In [ ]:
rng = np.random.default_rng(0)
x, u, eps = rng.uniform(-2, 2, 3), np.array([1.0, 0.4]), 1e-6
G_fd = np.stack([(dynamics(x + eps * e, u, dt) - dynamics(x - eps * e, u, dt)) / (2 * eps) for e in np.eye(3)], axis=1)
H_fd = np.stack([(measurement(x + eps * e, landmarks) - measurement(x - eps * e, landmarks)) / (2 * eps) for e in np.eye(3)], axis=1)
print("G error:", np.abs(jacobian_G(x, u, dt) - G_fd).max())
print("H error:", np.abs(jacobian_H(x, landmarks) - H_fd).max())

## Running the filter
For comparison, dead reckoning integrates the controls from the same initial estimate and ignores the measurements.

In [ ]:
mus, Sigmas = run_ekf(mu0, Sigma0, us, zs, landmarks, dt, R, Q)

dr = [mu0]
for u in us:
    dr.append(dynamics(dr[-1], u, dt))
dr = np.array(dr)

err = np.linalg.norm(mus[:, :2] - poses[:, :2], axis=1)
dr_err = np.linalg.norm(dr[:, :2] - poses[:, :2], axis=1)
print(f"EKF position error: mean {err.mean():.3f} m, final {err[-1]:.3f} m")
print(f"dead reckoning position error: final {dr_err[-1]:.3f} m")
print("final standard deviations (x, y, theta):", np.sqrt(np.diag(Sigmas[-1])).round(4))

## Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for k, name in enumerate(["x (m)", "y (m)", "theta (rad)"]):
    axes[k].plot(time, poses[:, k], lw=2, label="true")
    axes[k].plot(time, mus[:, k], ".", ms=3, label="EKF estimate")
    axes[k].set_xlabel("time (s)")
    axes[k].set_ylabel(name)
    axes[k].grid(True)
    axes[k].legend(fontsize=8)
axes[1].set_title("EKF estimates of the robot pose")
plt.tight_layout()
plt.savefig("plots/p3_ekf_time.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(poses[:, 0], poses[:, 1], lw=2, label="true path")
ax.plot(mus[:, 0], mus[:, 1], lw=1.5, label="EKF estimate")
ax.plot(dr[:, 0], dr[:, 1], "--", lw=1.5, label="dead reckoning")
ax.plot(landmarks[:, 0], landmarks[:, 1], "ks", ms=8, label="landmarks")
for t in range(0, len(mus), 10):
    plot_error_ellipse(ax, mus[t, :2], Sigmas[t][:2, :2])
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Robot path with 95% ellipses every 1 s")
ax.axis("equal")
ax.grid(True)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("plots/p3_ekf_map.png", dpi=120)
plt.show()